In [2]:
from datasets import load_dataset
import tqdm as notebook_tqdm
from collections import Counter
import torch
import torch.nn as nn
from torch.nn import functional as F
import tqdm
from tqdm import tqdm
from progress_table import ProgressTable


/home/xerneas/Coding/llm_test/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = load_dataset("Trelis/tiny-shakespeare")
train_data = dataset["train"]['Text']
test_data = dataset["test"]['Text']
char_count = sum(len(line) for line in train_data)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('train length:', char_count)
print(device)

train length: 1222354
cuda


In [4]:
chars = sorted(list(set("".join(train_data))))
vocab_size = len(chars)
print(vocab_size)
print("".join(chars))

65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [5]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hello"))
print(decode(encode("hello")))

[46, 43, 50, 50, 53]
hello


In [6]:
train = torch.tensor(encode("".join(train_data)), dtype=torch.long)
test = torch.tensor(encode("".join(test_data)), dtype=torch.long)
print(train.shape, train.dtype)
print(train[:100])
print(test.shape, test.dtype)
print(test[:100])


torch.Size([1222354]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])
torch.Size([119020]) torch.int64
tensor([32, 30, 13, 26, 21, 27, 10,  0, 21, 57,  1, 58, 46, 47, 57,  1, 63, 53,
        59, 56,  1, 57, 54, 43, 43, 42, 47, 52, 45, 12,  1, 52, 39, 63,  6,  1,
        58, 46, 43, 52,  6,  1, 45, 53, 53, 42,  1, 52, 47, 45, 46, 58,  1, 53,
        59, 56,  1, 54, 39, 56, 58,  2,  0,  0, 28, 17, 32, 30, 33, 15, 20, 21,
        27, 10,  0, 14, 43,  1, 54, 39, 58, 47, 43, 52, 58,  6,  1, 45, 43, 52,
        58, 50, 43, 51, 43, 52, 11,  1, 21,  1])


In [8]:
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
print(x.shape)
print(x.dtype)
print(x[0])

torch.Size([4, 8, 2])
torch.float32
tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])


In [ ]:
#manual bow test
xbow = torch.zeros(B,T,C)
for b in range(B):
    for t in range(T):
        xbow[b,t,:] = x[b, :t+1].mean(dim=0)
print(xbow.shape)
print(xbow[0])


torch.Size([4, 8, 2])
tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


In [ ]:
torch.manual_seed(42)
#using .tril to ensure that only previous tokens are considered
a = torch.tril(torch.ones(3,3))
#random test
a = a / torch.sum(a, dim=1, keepdim=True)
b = torch.randint(0, 10, (3,2)).float()
c = a @ b
print('a:', a)
print('b:', b)
print('c:', c)

a: tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b: tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c: tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [23]:
weights = torch.tril(torch.ones(T,T))
weights = weights / weights.sum(dim=1, keepdim=True)
print(weights)


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
